In [ ]:
import numpy as np
import keras
from keras import backend as K
from keras.models import Sequential
from keras.layers import Dense, Activation
from keras.optimizers import Adam, rmsprop, sgd
from keras.callbacks import EarlyStopping
import keras.backend as Ka
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing
from sklearn.model_selection import KFold
from numpy import array
from keras import optimizers
import pandas as pd

# 1. Importation et pr?paration des donn?es

In [ ]:
immo = pd.read_csv('../../data/ml/train_immo_100_more_k_12_10.csv')

In [ ]:
immo['mois'] = immo['mois'] - 12

### On vire les variables dont on a pas besoin

In [ ]:
immo2 = immo.copy()

In [ ]:
immo2 = immo2.drop(columns=['month'])

In [ ]:
immo2.sort_values(by=['id_bien','mois']).head(20)

In [ ]:
max_valeur = immo2.valeur.max()
min_valeur = immo2.valeur.min()

In [ ]:
database = immo2.values

Nous avons impl?menter deux fonctions :
- one_hot_encode_database()
- et normalize_database()

La premi?re fonction permet de transformer la variable cible en plusieurs variables binaires (format requis par Keras)
La deuxi?me permet de normaliser les variables entre 0 et 1.

### On normalise les donn?es quantitatives

In [ ]:

## One hot encode the columns of the database specified in list_id.
#
#  return a copy of the database, on hot encoded. 
def one_hot_encode_database(database, list_id):
    encoded_database = np.empty(shape=(database.shape[0],0), dtype=float)
    for id in range(database.shape[1]):
        if id in list_id:
            original_column = database[:, id]

            print(np.unique(original_column).shape)
            encoded_column = to_categorical(original_column)
            print(id)
            print(encoded_column.shape)
            encoded_database = np.column_stack((encoded_database,encoded_column))
        else:
            original_column = database[:, id]
            encoded_database = np.column_stack((encoded_database,original_column))
    return encoded_database

## Normalize between 0 and 1 each column of the database specified in list_id.
#
#  return a copy of the database, normalized. 
def normalize_database(database, list_id):
    encoded_database = database.copy()
    for  id in list_id :
        x = encoded_database[:, id]
        encoded_database[:, id] = (x-min(x))/(max(x)-min(x))
    return encoded_database


In [ ]:
res = pd.DataFrame(database)
res.head()

In [ ]:
cols_index = set(range(database.shape[1])) 
numerical_cols_index = cols_index  - {0,2,3,4,5}
numerical_cols_index = list(numerical_cols_index)

In [ ]:
database = normalize_database(database, numerical_cols_index)

In [ ]:
immo_train = pd.DataFrame(database)
immo_train.columns = immo2.columns
immo_train.head()

### Integer Encoding de ib_bien

In [ ]:
immo_train2 = immo_train.copy()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
immo_train2["id_bien"] = le.fit_transform(immo_train["id_bien"])

In [ ]:
immo_train2.head()

# 2. Construction du mod?le

La fonction build_NN permet de construire et de compiler un r?seau de neurones.

La fonction test_model permet fiter puis de tester/valider le mod?le avec une 5 fold validation. Elle retourne des indicateurs de performances ( matrice de confusion et coefficient de kappa de cohen ).

Pour trouver le mod?le final, il a fallu tester affiner les param?tres du mod?le avec une m?thode de "test and learn".

Voil? les param?tres qu'il a fallu ajuster:
- nombre de neurones par couche / nombre de couches
- learning rate
- epochs
- patience


In [ ]:
immo_train2.head()

In [ ]:
nb_type_bien = immo_train2['id_bien'].drop_duplicates().shape[0]

In [ ]:
nb_type_bien

In [ ]:
nb_features = immo_train2.shape[1]-3

In [ ]:
from keras.layers import Input, Embedding, LSTM, Dense
from keras.models import Model
from keras.layers import Flatten

kernel_initialiser = 'normal'

nb_features # contient le nombre de features sans le type de bien ( ie mois, lags et diffs )

# Headline input: meant to receive sequences of 100 integers, between 1 and 10000.
# Note that we can name any layer by passing it a "name" argument.
# notre entr?e ici est un simple entier donc shape = 1
main_input = Input(shape=(1,), dtype='int32', name='main_input')

# This embedding layer will encode the input sequence
# into a sequence of dense 512-dimensional vectors.
x = Embedding(output_dim=20, input_dim=nb_type_bien, input_length=1,embeddings_initializer=kernel_initialiser)(main_input)

#Argument must be a dense tensor: ((4750,), 512) - got shape [2], but wanted [2, 1].

auxiliary_input = Input(shape=(nb_features,), name='aux_input')

x = Flatten()(x)

x = keras.layers.concatenate([x, auxiliary_input])

x = Dense(30, activation='relu',kernel_initializer=kernel_initialiser)(x)

x = Dense(100, activation='relu',kernel_initializer=kernel_initialiser)(x)

x = Dense(100, activation='relu',kernel_initializer=kernel_initialiser)(x)

main_output = Dense(1, name='main_output', activation='relu',kernel_initializer='normal')(x)

model = Model(inputs=[main_input, auxiliary_input], outputs=[main_output])


In [ ]:
sgd = keras.optimizers.Adam(lr=0.001)
#sgd = keras.optimizers.RMSprop(lr=0.01)
model.compile(optimizer=sgd, loss="mean_squared_logarithmic_error")

# 3. Walk Forward Validation

La validation du mod?le est r?alis? grace ? une Walk Forward Validation. On entraine le mod?le sur 10 ann?es pour pr?dire l'ann?e suivante et on d?place la fen?tre d'ann?e en ann?e

In [ ]:
from sklearn.metrics import mean_squared_log_error
def rmsle(ytrue, ypred):
    return np.sqrt(mean_squared_log_error(ytrue, ypred))

In [ ]:
45-12

In [ ]:
30-12

In [ ]:
from sklearn.ensemble import RandomForestRegressor

mean_error = []
for mois in range(18,22):
    train = immo_train2[immo_train2['mois'] < mois]
    val = immo_train2[immo_train2['mois'] == mois]
    
    test_headline_data = val['id_bien']
    test_additional_data = val.drop(columns=['id_bien','valeur','mois'])
    test_labels = val['valeur']
    
    headline_data = train['id_bien']
    additional_data = train.drop(columns=['id_bien','valeur','mois'])
    labels = train['valeur']
    
    print("mois : " + str(mois+12))
    
    model.fit([headline_data, additional_data], [labels], epochs=4, batch_size=32)
    
    preds = np.squeeze(model.predict([test_headline_data, test_additional_data]))
    
    error = rmsle(test_labels, preds)
    print('Week %d - Error %.5f' % (mois+12, error))
    mean_error.append(error)
    print('Mean Error inter = %.5f' % np.mean(mean_error))
print('Mean Error Final = %.5f' % np.mean(mean_error))




In [ ]:
pd.DataFrame(labels).describe()

In [ ]:
error = rmsle(test_labels, preds)

In [ ]:
error

In [ ]:
type(test_labels.values)

In [ ]:
test_labels.values.shape

In [ ]:
preds.shape

In [ ]:
print(np.squeeze(preds))

In [ ]:
print(preds)

In [ ]:
    xtr, xts = train.drop(['valeur','id_bien'], axis=1), val.drop(['valeur','id_bien'], axis=1)
    ytr, yts = train['valeur'].values, val['valeur'].values
    
    mdl = RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=0)
    mdl.fit(xtr, ytr)
    
    p = mdl.predict(xts)
    
    error = rmsle(yts, p)
    print('Week %d - Error %.5f' % (mois, error))
    mean_error.append(error)
print('Mean Error = %.5f' % np.mean(mean_error))